# Kyuubi REST API by hand

`run()` hides the HTTP calls. Here they are one by one with `requests`, so you can see
exactly what Kyuubi 1.12 wants: `POST /api/v1/batches`, then poll the batch and read
`/localLog`. Two rules that trip people up: `className` is required for every SPARK batch
(Python sends `org.apache.spark.deploy.PythonRunner`) and `resource` is required even
when you upload the file.

In [ ]:
import kyuubi_lab as lab          # client/kyuubi_lab.py (on PYTHONPATH in the Jupyter image)
import pandas as pd
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
RID = lab.new_run_id()            # one id for every job in this notebook run
print("Kyuubi REST", lab.REST_URL, "| JDBC", f"{lab.JDBC_HOST}:{lab.JDBC_PORT}", "| user", lab.USER, "| run", RID)

In [ ]:
import requests, time, json
BASE = lab.REST_URL + "/api/v1/batches"
AUTH = (lab.USER, "")          # Kyuubi auth is NONE; the user still becomes the proxy user

## 1. Submit a Python file that already sits in RustFS (JSON body)

In [ ]:
body = {
    "batchType": "SPARK",
    "resource": "s3a://jobs/python/02_wordcount_s3.py",
    "className": "org.apache.spark.deploy.PythonRunner",
    "name": f"nb-rest-wordcount-{RID}",
    "args": ["s3a://data/input/sample.txt", f"s3a://data/output/wc_nb_{RID}"],
    "conf": {"spark.executor.memory": "1g"},
}
b = requests.post(BASE, json=body, auth=AUTH, timeout=60).json()
batch_id = b["id"]; b

## 2. Poll until it's done, printing new log lines

In [ ]:
offset = 0
while True:
    log = requests.get(f"{BASE}/{batch_id}/localLog", params={"from": offset, "size": 200}, auth=AUTH).json()
    for line in log.get("logRowSet", []):
        if "WORDCOUNT" in line or "ERROR" in line:
            print(line)
    offset += log.get("rowCount", 0)
    info = requests.get(f"{BASE}/{batch_id}", auth=AUTH).json()
    if info["state"] in ("FINISHED", "ERROR", "CANCELED"):
        break
    time.sleep(3)
{k: info.get(k) for k in ("id", "state", "appId", "appState")}

## 3. Upload a local file (multipart)
Part `batchRequest` carries the JSON; part `resourceFile` the script. This is what `py-upload` in the catalog does.

In [ ]:
import os, lab_runner
path = os.path.join(lab_runner.ROOT, "jobs", "python", "01_smoke_test.py")
req = {"batchType": "SPARK", "resource": "01_smoke_test.py",
       "className": "org.apache.spark.deploy.PythonRunner", "name": f"nb-rest-upload-{RID}"}
with open(path, "rb") as fh:
    files = [("batchRequest", (None, json.dumps(req), "application/json")),
             ("resourceFile", ("01_smoke_test.py", fh, "application/octet-stream"))]
    up = requests.post(BASE, files=files, auth=AUTH, timeout=60).json()
info = lab.kyuubi().follow(up["id"], out=lambda l: print(l) if "SMOKE" in l else None)
info["state"]

## 4. List recent batches

In [ ]:
recent = pd.DataFrame(lab.batches(10))
recent[[c for c in ("id", "name", "state", "appId", "appState") if c in recent]] if len(recent) else "no batches yet"

## 5. Kill a running batch
SparkPi with a big slice count runs long enough to kill it.

In [ ]:
k = lab.kyuubi()
from job_catalog import SPARK_VERSION
b = k.submit(f"/opt/spark/examples/jars/spark-examples_2.13-{SPARK_VERSION}.jar", f"nb-kill-me-{RID}",
             "org.apache.spark.examples.SparkPi", {}, ["5000"], remote=True)
time.sleep(5)
print("before:", k.get(b["id"])["state"])
k.kill(b["id"]); time.sleep(3)
print("after: ", k.get(b["id"])["state"])

## 6. A job that fails on purpose
Kyuubi must report `ERROR`, not `FINISHED`. `run()` raises when the state doesn't match what the catalog expects; this one expects ERROR, so it passes.

In [ ]:
r = lab.run('py-expected-failure', run_id=RID)
r['state']